# 🇧🇹 Bhutan Language Gap RAG Pipeline
### LangChain + LangGraph + ChromaDB

This notebook builds a full Retrieval-Augmented Generation (RAG) pipeline to assist with Bhutan language gap — bridging Dzongkha (the national language) and English through intelligent document retrieval and generation.

**Architecture Overview:**
```
Documents (Dzongkha/English) 
        ↓
   Text Splitter
        ↓
   Embeddings (HuggingFace multilingual)
        ↓
   ChromaDB (Vector Store)
        ↓
   LangGraph RAG Agent
     ├─ retrieve node
     ├─ grade_documents node
     ├─ generate node
     └─ rewrite_query node (fallback)
        ↓
   Final Answer
```

## 1. Install Dependencies

In [ ]:
!pip install -q \
    langchain \
    langchain-community \
    langchain-chroma \
    langchain-huggingface \
    langchain-anthropic \
    langgraph \
    chromadb \
    sentence-transformers \
    tiktoken \
    pypdf \
    python-dotenv

## 2. Environment Setup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

# Set your Anthropic API key
# Option A: set in .env file as ANTHROPIC_API_KEY=sk-...
# Option B: set directly below (not recommended for production)
# os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."

assert os.getenv("ANTHROPIC_API_KEY"), "❌ Please set your ANTHROPIC_API_KEY"
print("✅ API key loaded")

## 3. Imports

In [ ]:
# Core
import json
from pathlib import Path
from typing import List, Literal, TypedDict, Annotated
import operator

# LangChain — Document Loading & Splitting
from langchain_community.document_loaders import (
    TextLoader,
    PyPDFLoader,
    DirectoryLoader,
)
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.schema import Document

# LangChain — Embeddings
from langchain_huggingface import HuggingFaceEmbeddings

# LangChain — Vector Store
from langchain_chroma import Chroma

# LangChain — LLM (Claude)
from langchain_anthropic import ChatAnthropic

# LangChain — Prompts & Chains
from langchain.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser
from langchain_core.runnables import RunnablePassthrough

# LangGraph
from langgraph.graph import StateGraph, END, START
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver

print("✅ All imports successful")

## 4. Configuration

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME        = "claude-sonnet-4-20250514"   # Claude Sonnet 4 — best for multilingual
MODEL_TEMPERATURE = 0.2                           # Low temp for factual language tasks

# ── Embeddings ─────────────────────────────────────────────────────────────
# paraphrase-multilingual supports 50+ languages including Dzongkha-adjacent scripts
EMBEDDING_MODEL   = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# ── ChromaDB ───────────────────────────────────────────────────────────────
CHROMA_PERSIST_DIR = "./chroma_bhutan_db"
COLLECTION_NAME    = "bhutan_language_corpus"

# ── Chunking ───────────────────────────────────────────────────────────────
CHUNK_SIZE    = 800
CHUNK_OVERLAP = 150

# ── Retrieval ──────────────────────────────────────────────────────────────
TOP_K_DOCS = 5

print("✅ Configuration set")
print(f"   Model      : {MODEL_NAME}")
print(f"   Embeddings : {EMBEDDING_MODEL}")
print(f"   ChromaDB   : {CHROMA_PERSIST_DIR}")

## 5. Initialize LLM & Embeddings

In [ ]:
# ── Claude LLM ─────────────────────────────────────────────────────────────
llm = ChatAnthropic(
    model=MODEL_NAME,
    temperature=MODEL_TEMPERATURE,
    max_tokens=2048,
)

# ── Multilingual Embeddings ─────────────────────────────────────────────────
print("Loading multilingual embedding model (first run downloads ~120MB)...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)

print("✅ LLM and embeddings initialized")

## 6. Document Ingestion & ChromaDB Setup

Add your Bhutan language documents (Dzongkha texts, English translations, dictionaries, cultural docs) to the `./docs/` folder, or use the sample data below.

In [ ]:
# ── Sample Bhutan Language Corpus ───────────────────────────────────────────
# Replace or extend with real documents from ./docs/ folder

SAMPLE_DOCUMENTS = [
    Document(
        page_content="""Dzongkha (རྫོང་ཁ་) is the national language of Bhutan. 
        It belongs to the Tibeto-Burman language family and is closely related to Classical Tibetan. 
        Dzongkha is written in the Tibetan script, known as Uchen for formal writing. 
        The language is spoken by about 640,000 people primarily in western Bhutan.""",
        metadata={"source": "dzongkha_overview", "language": "en", "topic": "language_basics"}
    ),
    Document(
        page_content="""Common Dzongkha greetings and phrases:
        - Kuzu Zangpo La (ཀུ་ཟུ་བཟང་པོ་ལགས།) = Hello / Good day (formal greeting)
        - Kadrinchhe La (བཀའ་དྲིན་ཆེ་ལགས།) = Thank you
        - Choe Gateng Ba? (ཁྱོད་ག་ཏེང་བ།) = Where are you from?
        - Nga Bhutan Mi Yin (ང་འབྲུག་མི་ཡིན།) = I am Bhutanese
        - Gaki Bey? (ག་ཀི་བེ།) = How much does it cost?
        These phrases are essential for basic communication in Bhutan.""",
        metadata={"source": "dzongkha_phrases", "language": "bilingual", "topic": "greetings"}
    ),
    Document(
        page_content="""The language gap in Bhutan refers to the communication challenges between: 
        1. Dzongkha (national language) and English (official business/education language)
        2. Regional languages: Sharchopkha (east), Nepali/Lhotshamkha (south), and Dzongkha (west/center)
        3. Rural populations who speak local dialects vs urban educated populations
        Bridging this gap is critical for government services, healthcare, education, and economic inclusion.""",
        metadata={"source": "language_gap_analysis", "language": "en", "topic": "language_gap"}
    ),
    Document(
        page_content="""Bhutan's education system uses English as the medium of instruction from Class 1 onwards. 
        However, Dzongkha is a compulsory subject throughout schooling. 
        This dual-language approach creates a language gap for students from rural areas 
        who may only speak local dialects at home. 
        The Royal University of Bhutan has initiatives to preserve and digitize Dzongkha texts.""",
        metadata={"source": "education_policy", "language": "en", "topic": "education"}
    ),
    Document(
        page_content="""Dzongkha grammar basics:
        - Sentence structure: Subject-Object-Verb (SOV), unlike English Subject-Verb-Object
        - Example: 'Nga droe chu dang' (I water drink) = I drink water
        - Honorific forms exist for formal/respectful communication
        - Verbs change based on evidentiality (whether you witnessed the action or heard about it)
        - No grammatical gender in Dzongkha nouns
        - Tonal language with four tones affecting word meaning""",
        metadata={"source": "dzongkha_grammar", "language": "bilingual", "topic": "grammar"}
    ),
    Document(
        page_content="""Key Bhutan language resources and institutions:
        - Dzongkha Development Commission (DDC): Government body for Dzongkha promotion
        - Website: dzongkha.gov.bt — provides Dzongkha fonts, keyboards, learning materials
        - Rigsum Institute: Provides Dzongkha-English translation training
        - Pema Gatshel dictionary: Primary Dzongkha-English bilingual dictionary
        - Google has added limited Dzongkha support in 2019
        - Unicode support for Tibetan script (which Dzongkha uses) added in Unicode 1.0""",
        metadata={"source": "language_resources", "language": "en", "topic": "resources"}
    ),
]

print(f"✅ Loaded {len(SAMPLE_DOCUMENTS)} sample documents")

In [ ]:
# ── Optional: Load from ./docs/ folder ─────────────────────────────────────
# Uncomment to load real PDFs or text files

# docs_path = Path("./docs")
# docs_path.mkdir(exist_ok=True)
#
# loader = DirectoryLoader(
#     str(docs_path),
#     glob="**/*.txt",
#     loader_cls=TextLoader,
#     loader_kwargs={"encoding": "utf-8"},
#     show_progress=True,
# )
# file_docs = loader.load()
# SAMPLE_DOCUMENTS.extend(file_docs)
# print(f"Total documents after loading: {len(SAMPLE_DOCUMENTS)}")

In [ ]:
# ── Text Splitting ──────────────────────────────────────────────────────────
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", "!", "?", " ", ""],
)

splits = splitter.split_documents(SAMPLE_DOCUMENTS)
print(f"✅ Split into {len(splits)} chunks (from {len(SAMPLE_DOCUMENTS)} documents)")
print(f"   Avg chunk size: {sum(len(s.page_content) for s in splits)//len(splits)} chars")

In [ ]:
# ── ChromaDB Vector Store ───────────────────────────────────────────────────
print("Building ChromaDB vector store...")

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=CHROMA_PERSIST_DIR,
)

retriever = vectorstore.as_retriever(
    search_type="mmr",                          # Maximal Marginal Relevance for diversity
    search_kwargs={"k": TOP_K_DOCS, "fetch_k": 20},
)

print(f"✅ ChromaDB ready — collection: '{COLLECTION_NAME}'")
print(f"   Persisted at: {CHROMA_PERSIST_DIR}")
print(f"   Total vectors: {vectorstore._collection.count()}")

## 7. LangGraph RAG Pipeline

We build an **agentic RAG graph** with these nodes:

| Node | Purpose |
|------|---------|
| `retrieve` | Fetch relevant chunks from ChromaDB |
| `grade_documents` | Check if retrieved docs are relevant |
| `generate` | Produce final answer using Claude |
| `rewrite_query` | Reformulate query if docs aren't relevant |

```
START → retrieve → grade_documents
                       ↓           ↘
                    generate    rewrite_query → retrieve (loop)
                       ↓
                      END
```

In [ ]:
# ── Graph State ─────────────────────────────────────────────────────────────
class RAGState(TypedDict):
    question: str                    # Original user question
    rewritten_question: str          # Query-rewritten version (if needed)
    documents: List[Document]        # Retrieved documents
    generation: str                  # Final answer
    relevance_scores: List[str]      # Grading results per doc
    rewrite_count: int               # Prevent infinite rewrite loops

print("✅ RAGState defined")

In [ ]:
# ── Prompt Templates ────────────────────────────────────────────────────────

# Grader: Is document relevant to question?
GRADER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a relevance grader for a Bhutan language assistance system.
Assess whether a retrieved document chunk is relevant to answer the user's question.
Consider both Dzongkha and English content. Output ONLY valid JSON.

Output format: {{"score": "yes"}} or {{"score": "no"}}"""),
    ("human", "Retrieved document:\n{document}\n\nUser question: {question}"),
])

# Query rewriter: improve the query
REWRITER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a query optimizer for a Bhutan language RAG system.
Rewrite the user's question to be more specific and retrieval-friendly.
Consider Dzongkha, Bhutanese culture, and language gap contexts.
Return ONLY the rewritten question, nothing else."""),
    ("human", "Original question: {question}\n\nRewrite it to improve retrieval:"),
])

# Main RAG generator
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are an expert assistant specializing in Bhutan's language landscape — 
including Dzongkha (རྫོང་ཁ་), English, Sharchopkha, and regional dialects.

Your role is to bridge language gaps by:
- Providing accurate translations and explanations
- Explaining linguistic and cultural context
- Helping users understand Dzongkha grammar, vocabulary, and usage
- Supporting communication across Bhutan's diverse language communities

Use the provided context to answer. If the context doesn't contain enough information, 
say so clearly and provide what general knowledge you have.

Context:
{context}"""),
    ("human", "{question}"),
])

print("✅ Prompt templates ready")

In [ ]:
# ── Chains ──────────────────────────────────────────────────────────────────

grader_chain  = GRADER_PROMPT  | llm | JsonOutputParser()
rewriter_chain = REWRITER_PROMPT | llm | StrOutputParser()
rag_chain      = RAG_PROMPT    | llm | StrOutputParser()

print("✅ LangChain chains built")

In [ ]:
# ── Node Functions ───────────────────────────────────────────────────────────

MAX_REWRITES = 2   # Safety cap to prevent infinite loops


def retrieve(state: RAGState) -> RAGState:
    """Retrieve documents from ChromaDB."""
    query = state.get("rewritten_question") or state["question"]
    print(f"\n🔍 Retrieving for: '{query[:80]}...' " if len(query) > 80 else f"\n🔍 Retrieving for: '{query}'")
    
    docs = retriever.invoke(query)
    print(f"   → Found {len(docs)} document chunks")
    
    return {**state, "documents": docs}


def grade_documents(state: RAGState) -> RAGState:
    """Grade each retrieved document for relevance."""
    print("\n📋 Grading document relevance...")
    question  = state.get("rewritten_question") or state["question"]
    documents = state["documents"]
    
    relevant_docs = []
    scores = []
    
    for i, doc in enumerate(documents):
        try:
            result = grader_chain.invoke({
                "question": question,
                "document": doc.page_content,
            })
            score = result.get("score", "no")
        except Exception:
            score = "yes"   # Default to include on parse error
        
        scores.append(score)
        if score == "yes":
            relevant_docs.append(doc)
        print(f"   Doc {i+1}: {'✅ relevant' if score == 'yes' else '❌ not relevant'} — {doc.metadata.get('source', 'unknown')}")
    
    return {**state, "documents": relevant_docs, "relevance_scores": scores}


def rewrite_query(state: RAGState) -> RAGState:
    """Rewrite the query to improve retrieval."""
    rewrite_count = state.get("rewrite_count", 0) + 1
    print(f"\n✏️  Rewriting query (attempt {rewrite_count}/{MAX_REWRITES})...")
    
    original = state["question"]
    rewritten = rewriter_chain.invoke({"question": original})
    print(f"   Original : {original}")
    print(f"   Rewritten: {rewritten}")
    
    return {**state, "rewritten_question": rewritten, "rewrite_count": rewrite_count}


def generate(state: RAGState) -> RAGState:
    """Generate final answer using Claude + retrieved context."""
    print("\n💬 Generating answer...")
    question  = state.get("rewritten_question") or state["question"]
    documents = state["documents"]
    
    context = "\n\n---\n\n".join([
        f"[Source: {doc.metadata.get('source', 'unknown')} | Topic: {doc.metadata.get('topic', 'general')}]\n{doc.page_content}"
        for doc in documents
    ]) if documents else "No relevant documents found in the knowledge base."
    
    answer = rag_chain.invoke({"question": question, "context": context})
    return {**state, "generation": answer}


# ── Conditional Edge ─────────────────────────────────────────────────────────

def decide_after_grading(state: RAGState) -> Literal["generate", "rewrite_query"]:
    """Route to generate if docs are relevant, else rewrite query."""
    rewrite_count = state.get("rewrite_count", 0)
    has_relevant  = len(state["documents"]) > 0
    
    if has_relevant or rewrite_count >= MAX_REWRITES:
        print("   → Routing to: GENERATE")
        return "generate"
    else:
        print("   → Routing to: REWRITE QUERY")
        return "rewrite_query"


print("✅ Node functions defined")

In [ ]:
# ── Build LangGraph ──────────────────────────────────────────────────────────

workflow = StateGraph(RAGState)

# Add nodes
workflow.add_node("retrieve",        retrieve)
workflow.add_node("grade_documents", grade_documents)
workflow.add_node("generate",        generate)
workflow.add_node("rewrite_query",   rewrite_query)

# Add edges
workflow.add_edge(START,             "retrieve")
workflow.add_edge("retrieve",        "grade_documents")
workflow.add_conditional_edges(
    "grade_documents",
    decide_after_grading,
    {
        "generate":      "generate",
        "rewrite_query": "rewrite_query",
    }
)
workflow.add_edge("rewrite_query",   "retrieve")   # Loop back
workflow.add_edge("generate",        END)

# Compile with in-memory checkpointer (enables multi-turn conversation)
memory   = MemorySaver()
rag_app  = workflow.compile(checkpointer=memory)

print("✅ LangGraph RAG pipeline compiled")
print("\nGraph nodes:", list(workflow.nodes.keys()))

## 8. Visualize the Graph

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(rag_app.get_graph().draw_mermaid_png()))
except Exception as e:
    # Fallback: print Mermaid diagram source
    print("Graph (Mermaid source):")
    print(rag_app.get_graph().draw_mermaid())

## 9. Query Helper Function

In [ ]:
def ask_bhutan_rag(question: str, thread_id: str = "default") -> str:
    """
    Query the Bhutan Language RAG pipeline.
    
    Args:
        question:  Your question in English or Dzongkha
        thread_id: Conversation thread ID for multi-turn memory
    
    Returns:
        str: Generated answer
    """
    print("=" * 60)
    print(f"❓ Question: {question}")
    print("=" * 60)
    
    initial_state: RAGState = {
        "question":           question,
        "rewritten_question": "",
        "documents":          [],
        "generation":         "",
        "relevance_scores":   [],
        "rewrite_count":      0,
    }
    
    config = {"configurable": {"thread_id": thread_id}}
    
    result = rag_app.invoke(initial_state, config=config)
    
    print("\n" + "=" * 60)
    print("📖 ANSWER:")
    print("=" * 60)
    print(result["generation"])
    print("=" * 60)
    print(f"📚 Sources used: {[d.metadata.get('source') for d in result['documents']]}")
    
    return result["generation"]


print("✅ ask_bhutan_rag() helper ready")

## 10. Run Example Queries

In [ ]:
# Query 1: Basic Dzongkha greeting
answer1 = ask_bhutan_rag(
    "How do I say 'hello' and 'thank you' in Dzongkha?",
    thread_id="session_1"
)

In [ ]:
# Query 2: Language gap understanding
answer2 = ask_bhutan_rag(
    "What is the language gap problem in Bhutan and why does it matter?",
    thread_id="session_1"
)

In [ ]:
# Query 3: Grammar question
answer3 = ask_bhutan_rag(
    "How is Dzongkha sentence structure different from English?",
    thread_id="session_1"
)

In [ ]:
# Query 4: Resources
answer4 = ask_bhutan_rag(
    "Where can I find resources to learn Dzongkha online?",
    thread_id="session_1"
)

## 11. Add New Documents to ChromaDB

Extend the knowledge base at any time:

In [ ]:
def add_documents_to_vectorstore(texts: List[str], metadatas: List[dict] = None):
    """
    Add new text documents to the ChromaDB vector store.
    
    Args:
        texts:     List of text strings to add
        metadatas: Optional list of metadata dicts
    """
    if metadatas is None:
        metadatas = [{"source": f"manual_entry_{i}"} for i in range(len(texts))]
    
    new_docs = [
        Document(page_content=text, metadata=meta)
        for text, meta in zip(texts, metadatas)
    ]
    
    new_splits = splitter.split_documents(new_docs)
    vectorstore.add_documents(new_splits)
    
    print(f"✅ Added {len(new_splits)} chunks from {len(texts)} new documents")
    print(f"   Total vectors in DB: {vectorstore._collection.count()}")


# Example: Add a new Dzongkha vocabulary entry
add_documents_to_vectorstore(
    texts=[
        """Dzongkha numbers (1-10):
        1=Chi, 2=Nyi, 3=Sum, 4=Zhi, 5=Nga,
        6=Drug, 7=Dün, 8=Gyä, 9=Gu, 10=Chu.
        These are essential for markets, time-telling, and daily conversation in Bhutan."""
    ],
    metadatas=[{"source": "dzongkha_numbers", "topic": "vocabulary", "language": "bilingual"}]
)

## 12. Reload Existing ChromaDB (for future sessions)

In [ ]:
# If you've already built the DB in a previous session, reload it here:

def load_existing_vectorstore():
    """Load an existing persisted ChromaDB without re-embedding."""
    vs = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=CHROMA_PERSIST_DIR,
    )
    print(f"✅ Loaded existing ChromaDB: {vs._collection.count()} vectors")
    return vs

# Uncomment to use:
# vectorstore = load_existing_vectorstore()
# retriever   = vectorstore.as_retriever(search_type="mmr", search_kwargs={"k": TOP_K_DOCS})

## 13. Pipeline Summary

| Component | Choice | Purpose |
|-----------|--------|---------|
| **LLM** | Claude Sonnet 4 | Multilingual generation & reasoning |
| **Embeddings** | `paraphrase-multilingual-MiniLM-L12-v2` | Cross-lingual semantic search |
| **Vector DB** | ChromaDB (local) | Persistent document retrieval |
| **Retrieval** | MMR (Maximal Marginal Relevance) | Diverse, non-redundant chunks |
| **Orchestration** | LangGraph | Stateful agentic RAG with fallbacks |
| **Memory** | LangGraph MemorySaver | Multi-turn conversation threads |

### Pipeline Flow
```
User Query
    ↓
[retrieve] → ChromaDB MMR search (top-5 chunks)
    ↓
[grade_documents] → Claude checks each doc for relevance
    ↓ (relevant found)         ↓ (none relevant + rewrites < 2)
[generate]              [rewrite_query] → [retrieve] ↺
    ↓
Final Answer (Claude Sonnet 4 + context)
```

### To extend this pipeline:
- Add PDF/Word documents to `./docs/` and use `DirectoryLoader`
- Swap embeddings for a Tibetan-script-specific model when available
- Add a translation node to auto-translate Dzongkha queries to English before retrieval
- Connect a web search tool for real-time Bhutan language resources